# CHB-MIT Seizure Onset Boundary Evaluation

Purpose: Retrain ACBL + Isolation v1 on CHB-MIT and evaluate whether boundary head activations align with annotated seizure onset times.

Metrics:
  1. Onset detection latency (seconds from onset to boundary peak)
  2. Boundary activation seizure vs baseline (separation)
  3. False boundary rate
  4. PELT baseline comparison
  5. Standard classification metrics

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from pathlib import Path
import h5py
import json
import re
from collections import defaultdict
from typing import Dict, Optional, Tuple
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    roc_auc_score, confusion_matrix, balanced_accuracy_score
)
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [2]:
# CONFIG

class Config:
    h5_path = Path("data/processed/chbmit_combined_seizure_detection.h5")
    raw_dir = Path("data/raw/chb-mit")
    model_save_dir = Path("models")
    figures_dir = Path("figures/onset_eval")

    task = "seizure_detection"
    n_classes = 2
    n_channels = 17
    sfreq = 100
    epoch_duration = 30.0
    n_samples = int(sfreq * epoch_duration)  # 3000
    n_context = 3
    embed_dim = 128
    n_layers = 4
    dropout = 0.1
    contrast_scales = (1, 4, 16)
    cp_hidden = 64

    # Computed from PatchEmbedding: (3000 - 75) // 15 + 1 = 196
    tokens_per_epoch = 196
    total_tokens = tokens_per_epoch * n_context  # 588
    seconds_per_token = epoch_duration / tokens_per_epoch  # ~0.153s

    # Training
    batch_size = 32
    lr = 1e-4
    weight_decay = 1e-4
    n_train_epochs = 50
    warmup_epochs = 3
    formation_epochs = 12
    patience = 12
    acbl_weight = 0.3
    cls_weight = 1.0

    seed = 42

Config.model_save_dir.mkdir(parents=True, exist_ok=True)
Config.figures_dir.mkdir(parents=True, exist_ok=True)

In [3]:
# MODEL COMPONENTS
# All components from tier 2 experiments, verbatim.

class PatchEmbedding(nn.Module):
    def __init__(self, n_channels=17, n_samples=3000, embed_dim=128,
                 temporal_kernel=25, pool_kernel=75, pool_stride=15,
                 dropout=0.1):
        super().__init__()
        self.temporal_conv = nn.Sequential(
            nn.Conv2d(1, 40, (1, temporal_kernel),
                      padding=(0, temporal_kernel // 2)),
            nn.BatchNorm2d(40), nn.GELU())
        self.spatial_conv = nn.Sequential(
            nn.Conv2d(40, 40, (n_channels, 1)),
            nn.BatchNorm2d(40), nn.GELU())
        self.pool = nn.AvgPool2d((1, pool_kernel), stride=(1, pool_stride))
        self.projection = nn.Sequential(
            nn.Conv2d(40, embed_dim, (1, 1)), nn.Dropout(dropout))
        self.seq_len = (n_samples - pool_kernel) // pool_stride + 1

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.temporal_conv(x)
        x = self.spatial_conv(x)
        x = self.pool(x)
        x = self.projection(x)
        return x.squeeze(2).permute(0, 2, 1)


class MultiResolutionEncoder(nn.Module):
    def __init__(self, n_channels=17, n_samples=3000,
                 embed_dim=128, dropout=0.1):
        super().__init__()
        self.enc_100 = PatchEmbedding(n_channels, n_samples, embed_dim,
                                       dropout=dropout)
        self.enc_50 = PatchEmbedding(n_channels, n_samples // 2, embed_dim,
                                      dropout=dropout)
        self.enc_25 = PatchEmbedding(n_channels, n_samples // 4, embed_dim,
                                      dropout=dropout)
        self.merge = nn.Sequential(
            nn.Linear(embed_dim * 3, embed_dim), nn.GELU(),
            nn.Dropout(dropout))
        self.seq_len_100 = self.enc_100.seq_len

    def forward(self, x):
        e100 = self.enc_100(x)
        e50 = self.enc_50(x[:, :, ::2])
        e25 = self.enc_25(x[:, :, ::4])
        T = e100.shape[1]
        e50 = F.interpolate(e50.permute(0, 2, 1), size=T,
                            mode='linear', align_corners=False).permute(0, 2, 1)
        e25 = F.interpolate(e25.permute(0, 2, 1), size=T,
                            mode='linear', align_corners=False).permute(0, 2, 1)
        return self.merge(torch.cat([e100, e50, e25], dim=-1))


class ContrastiveBoundaryModule(nn.Module):
    def __init__(self, embed_dim=128, hidden_dim=64,
                 scales=(1, 4, 16), dropout=0.1):
        super().__init__()
        self.scales = scales
        self.projections = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, hidden_dim), nn.GELU(),
                nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim))
            for _ in scales])
        self.fusion = nn.Sequential(
            nn.Linear(len(scales), len(scales) * 2), nn.GELU(),
            nn.Linear(len(scales) * 2, 1))
        self.temperature = nn.Parameter(torch.tensor(1.0))

    def _contrast(self, x, proj, offset):
        B, T, D = x.shape
        h = F.normalize(proj(x), dim=-1)
        if offset < T:
            h_shift = torch.roll(h, -offset, dims=1)
            h_shift[:, -offset:, :] = h[:, -offset:, :]
            sim = (h * h_shift).sum(dim=-1)
            c = 1.0 - (sim + 1.0) / 2.0
            c = torch.sigmoid(
                (c - 0.5) * self.temperature.abs().clamp(min=0.1))
        else:
            c = torch.zeros(B, T, device=x.device)
        return c

    def forward(self, x):
        per_scale = [self._contrast(x, p, o)
                     for p, o in zip(self.projections, self.scales)]
        stacked = torch.stack(per_scale, dim=-1)
        fused = torch.sigmoid(self.fusion(stacked).squeeze(-1))
        consistency = sum(F.mse_loss(ps, fused.detach())
                         for ps in per_scale) / len(per_scale)
        return {'boundaries': fused, 'per_scale': per_scale,
                'boundary_loss': 0.01 * consistency}


class OriginalRegimeMask(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, boundaries):
        cum = torch.cumsum(boundaries, dim=1)
        same = torch.exp(-torch.abs(cum.unsqueeze(2) - cum.unsqueeze(1)))
        return same, 1.0 - same


class RegimeStructuredAttention(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, dropout=0.1, regime_mask_module=None):
        super().__init__()
        self.n_heads = n_intra + n_inter + n_cross
        self.head_dim = embed_dim // self.n_heads
        self.n_intra = n_intra
        self.n_inter = n_inter
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.proj_drop = nn.Dropout(dropout)
        self.regime_mask = regime_mask_module or OriginalRegimeMask()

    def forward(self, x, boundaries, return_attention=False):
        B, T, D = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale

        same, cross = self.regime_mask(boundaries)
        same = same.unsqueeze(1)
        cross = cross.unsqueeze(1)

        h1 = self.n_intra
        h2 = h1 + self.n_inter
        mask = torch.ones_like(attn)
        mask[:, :h1] = same.expand(B, self.n_intra, T, T)
        mask[:, h1:h2] = cross.expand(B, self.n_inter, T, T)
        attn = attn + torch.log(mask + 1e-6)

        w = F.softmax(attn, dim=-1)
        w = self.attn_drop(w)
        out = (w @ v).transpose(1, 2).reshape(B, T, D)
        out = self.proj_drop(self.out_proj(out))

        if return_attention:
            return out, w
        return out


class NeuroStateBlock(nn.Module):
    def __init__(self, embed_dim=128, n_intra=4, n_inter=2,
                 n_cross=2, mlp_ratio=4.0, dropout=0.1,
                 regime_mask_module=None):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = RegimeStructuredAttention(
            embed_dim, n_intra, n_inter, n_cross, dropout,
            regime_mask_module)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout))

    def forward(self, x, boundaries, return_attention=False):
        if return_attention:
            attn_out, attn_w = self.attn(
                self.norm1(x), boundaries, return_attention=True)
            x = x + attn_out
            x = x + self.mlp(self.norm2(x))
            return x, attn_w
        x = x + self.attn(self.norm1(x), boundaries)
        x = x + self.mlp(self.norm2(x))
        return x

In [4]:
# FULL MODEL

class MultiResContrastiveNeuroState(nn.Module):
    def __init__(self, n_channels=17, n_samples=3000, n_classes=2,
                 embed_dim=128, n_layers=4, dropout=0.1,
                 contrast_scales=(1, 4, 16), cp_hidden=64,
                 n_context_epochs=3):
        super().__init__()
        self.n_classes = n_classes
        self.n_context = n_context_epochs

        self.mr_encoder = MultiResolutionEncoder(
            n_channels, n_samples, embed_dim, dropout)

        self.tokens_per_epoch = self.mr_encoder.seq_len_100
        total_tokens = self.tokens_per_epoch * n_context_epochs

        self.pos_embed = nn.Parameter(
            torch.randn(1, total_tokens, embed_dim) * 0.02)
        self.pos_drop = nn.Dropout(dropout)
        self.epoch_embed = nn.Parameter(
            torch.randn(1, n_context_epochs, 1, embed_dim) * 0.02)

        self.changepoint_module = ContrastiveBoundaryModule(
            embed_dim, cp_hidden, contrast_scales, dropout)

        self.blocks = nn.ModuleList([
            NeuroStateBlock(embed_dim, dropout=dropout)
            for _ in range(n_layers)])

        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(embed_dim // 2, n_classes))

        self.apply(self._init_weights)
        n_params = sum(p.numel() for p in self.parameters())
        print(f"MultiResContrastiveNeuroState: {n_params/1e6:.2f}M params, "
              f"{total_tokens} tokens ({self.tokens_per_epoch}/epoch)")

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

In [5]:
# FORWARD WITH INTERMEDIATES (for ACBL + gradient isolation)

def forward_with_intermediates(model, x, detach_boundaries=False):
    """
    Forward pass exposing boundary probs and encoder hidden states.
    When detach_boundaries=True, boundaries are detached before entering
    the transformer blocks so classification cannot send gradients to
    the boundary head.
    """
    B, N, C, T = x.shape

    epoch_embs = []
    for i in range(N):
        emb = model.mr_encoder(x[:, i])
        emb = emb + model.epoch_embed[:, i]
        epoch_embs.append(emb)

    full_seq = torch.cat(epoch_embs, dim=1)
    full_seq = model.pos_drop(full_seq + model.pos_embed)

    encoder_h = full_seq

    cp_out = model.changepoint_module(full_seq)
    boundaries = cp_out['boundaries']
    boundary_loss = cp_out['boundary_loss']

    boundaries_for_attn = boundaries.detach() if detach_boundaries else boundaries

    regime_attention = None
    for i, block in enumerate(model.blocks):
        if i == len(model.blocks) - 1:
            full_seq, attn_w = block(
                full_seq, boundaries_for_attn, return_attention=True)
            regime_attention = attn_w
        else:
            full_seq = block(full_seq, boundaries_for_attn)

    full_seq = model.norm(full_seq)
    tpe = model.tokens_per_epoch
    start = tpe * (N // 2)
    end = start + tpe
    pooled = full_seq[:, start:end, :].mean(dim=1)
    logits = model.head(pooled)

    return {
        'logits': logits,
        'boundary_loss': boundary_loss,
        'boundaries': boundaries,
        'encoder_h': encoder_h,
        'boundary_probs': boundaries,
        'regime_attention': regime_attention,
    }

In [6]:
# ACBL LOSSES

class PseudoBoundaryLoss(nn.Module):
    """Prong 1: pseudo-boundary targets from representation contrast."""
    def __init__(self, temperature=2.0, tokens_per_epoch=196, n_epochs=3):
        super().__init__()
        self.temperature = temperature
        self.tpe = tokens_per_epoch
        self.n_epochs = n_epochs

    def forward(self, boundary_probs, encoder_h):
        B, T, D = encoder_h.shape
        h_norm = F.normalize(encoder_h, dim=-1)
        sim = (h_norm[:, :-1] * h_norm[:, 1:]).sum(dim=-1)
        dissim = 1.0 - sim
        pseudo_targets = torch.sigmoid(dissim * self.temperature)
        loss = F.binary_cross_entropy(
            boundary_probs[:, :-1], pseudo_targets.detach())
        return loss


class AttentionPriorLoss(nn.Module):
    """Prong 2: Gaussian boundary targets at label transitions + attention KL."""
    def __init__(self, tokens_per_epoch=196, n_epochs=3,
                 sigma=5.0, attn_prior_weight=0.5):
        super().__init__()
        self.tpe = tokens_per_epoch
        self.n_epochs = n_epochs
        self.sigma = sigma
        self.attn_prior_weight = attn_prior_weight

    def forward(self, boundary_probs, regime_attention, labels):
        B, T = boundary_probs.shape
        device = boundary_probs.device
        tpe = self.tpe

        # Gaussian targets at epoch boundaries where labels change
        targets = torch.zeros(B, T, device=device)
        mask = torch.zeros(B, device=device)
        n_transitions = 0

        for b_idx in range(B):
            has_transition = False
            for ep in range(self.n_epochs - 1):
                if labels[b_idx, ep] != labels[b_idx, ep + 1]:
                    boundary_token = (ep + 1) * tpe
                    positions = torch.arange(T, device=device, dtype=torch.float32)
                    gaussian = torch.exp(
                        -0.5 * ((positions - boundary_token) / self.sigma) ** 2)
                    targets[b_idx] = torch.max(targets[b_idx], gaussian)
                    has_transition = True
                    n_transitions += 1
            if has_transition:
                mask[b_idx] = 1.0

        if mask.sum() > 0:
            bnd_target_loss = F.binary_cross_entropy(
                boundary_probs, targets, reduction='none')
            bnd_target_loss = (bnd_target_loss.mean(dim=1) * mask).sum() / mask.sum()
        else:
            bnd_target_loss = torch.tensor(0.0, device=device)

        # Attention KL prior
        attn_prior_loss = torch.tensor(0.0, device=device)
        if regime_attention is not None and mask.sum() > 0:
            if regime_attention.dim() == 4:
                attn = regime_attention.mean(dim=1)
            else:
                attn = regime_attention
            attn = attn / (attn.sum(dim=-1, keepdim=True) + 1e-8)

            # Build block-diagonal prior from labels
            prior = torch.zeros(B, T, T, device=device)
            for b_idx in range(B):
                for ep in range(self.n_epochs):
                    s = ep * tpe
                    e = min((ep + 1) * tpe, T)
                    prior[b_idx, s:e, s:e] = 1.0
            prior = prior / (prior.sum(dim=-1, keepdim=True) + 1e-8)

            kl = prior * (torch.log(prior.clamp(min=1e-8))
                          - torch.log(attn.clamp(min=1e-8)))
            kl_per_sample = kl.sum(dim=-1).mean(dim=-1)
            attn_prior_loss = (kl_per_sample * mask).sum() / mask.sum()

        results = {
            'boundary_target_loss': bnd_target_loss,
            'attention_prior_loss': attn_prior_loss,
            'n_transitions': n_transitions,
            'total': bnd_target_loss + self.attn_prior_weight * attn_prior_loss,
        }
        return results


class ACBLLoss(nn.Module):
    def __init__(self, tokens_per_epoch=196, n_epochs=3,
                 pseudo_weight=0.3, prior_weight=0.5,
                 pseudo_temperature=2.0, pseudo_temp_min=0.5,
                 pseudo_temp_anneal_epochs=15,
                 sigma=5.0, attn_prior_weight=0.5):
        super().__init__()
        self.pseudo_weight = pseudo_weight
        self.prior_weight = prior_weight
        self.pseudo_temp_min = pseudo_temp_min
        self.pseudo_temp_anneal_epochs = pseudo_temp_anneal_epochs
        self.pseudo_temperature_init = pseudo_temperature

        self.prong1 = PseudoBoundaryLoss(
            pseudo_temperature, tokens_per_epoch, n_epochs)
        self.prong2 = AttentionPriorLoss(
            tokens_per_epoch, n_epochs, sigma, attn_prior_weight)

    def anneal_temperature(self, epoch):
        if self.pseudo_temp_anneal_epochs <= 0:
            return
        progress = min(epoch / self.pseudo_temp_anneal_epochs, 1.0)
        self.prong1.temperature = self.pseudo_temperature_init - progress * (
            self.pseudo_temperature_init - self.pseudo_temp_min)

    def forward(self, boundary_probs, encoder_h, regime_attention,
                labels, epoch=0):
        self.anneal_temperature(epoch)
        pseudo_loss = self.prong1(boundary_probs, encoder_h)
        prior_results = self.prong2(boundary_probs, regime_attention, labels)
        total = (self.pseudo_weight * pseudo_loss
                 + self.prior_weight * prior_results['total'])
        return {
            'acbl_total': total,
            'pseudo_boundary_loss': pseudo_loss,
            'boundary_target_loss': prior_results['boundary_target_loss'],
            'attention_prior_loss': prior_results['attention_prior_loss'],
            'n_transitions': prior_results['n_transitions'],
        }

In [7]:
# DATA LOADING

def load_chbmit_data(h5_path):
    with h5py.File(h5_path, 'r') as f:
        epochs = f['epochs'][:]
        labels = f['labels'][:]
        subject_ids = np.array([s.decode() for s in f['subject_ids'][:]])
        sfreq = f.attrs['sfreq']
    print(f"Loaded: {epochs.shape[0]} epochs, {epochs.shape[1]} ch, {sfreq} Hz")
    print(f"  Seizure: {np.sum(labels == 1)}, Normal: {np.sum(labels == 0)}")
    subjects = np.unique(subject_ids)
    subj_sz = {}
    for s in subjects:
        subj_sz[s] = int(np.sum(labels[subject_ids == s] == 1))
    for s in sorted(subj_sz, key=subj_sz.get, reverse=True):
        print(f"    {s}: {subj_sz[s]} seizure epochs")
    return epochs, labels, subject_ids, subj_sz


def round_robin_split(subj_sz, seed=42):
    sorted_s = sorted(subj_sz.keys(), key=lambda s: subj_sz[s], reverse=True)
    splits = {'train': [], 'val': [], 'test': []}
    pattern = ['train', 'train', 'val', 'test'] * 4
    for i, s in enumerate(sorted_s):
        splits[pattern[i % len(pattern)]].append(s)
    for name, subjs in splits.items():
        total = sum(subj_sz[s] for s in subjs)
        print(f"  {name}: {subjs} ({total} seizure epochs)")
    return splits


class CHBMITOnsetDataset(Dataset):
    def __init__(self, epochs, labels, subject_ids, subject_list,
                 n_context=3):
        mask = np.isin(subject_ids, subject_list)
        self.epochs = epochs[mask]
        self.labels = labels[mask]
        self.subject_ids = subject_ids[mask]
        self.n_ctx = n_context

        self.windows = []
        for subj in subject_list:
            subj_mask = self.subject_ids == subj
            subj_idx = np.where(subj_mask)[0]
            if len(subj_idx) < n_context:
                continue
            for start in range(len(subj_idx) - n_context + 1):
                group = subj_idx[start:start + n_context]
                if np.all(np.diff(group) == 1):
                    center = n_context // 2
                    center_label = self.labels[group[center]]
                    epoch_labels = [int(self.labels[g]) for g in group]
                    self.windows.append({
                        'indices': group,
                        'label': int(center_label),
                        'epoch_labels': epoch_labels,
                        'subject': subj,
                    })

        win_labels = [w['label'] for w in self.windows]
        print(f"  {len(self.windows)} windows "
              f"({sum(win_labels)} seizure, "
              f"{len(win_labels) - sum(win_labels)} normal)")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        x = np.stack([self.epochs[i] for i in w['indices']], axis=0)
        return {
            'x': torch.tensor(x, dtype=torch.float32),
            'label': torch.tensor(w['label'], dtype=torch.long),
            'epoch_labels': torch.tensor(w['epoch_labels'], dtype=torch.long),
            'indices': w['indices'],
            'subject': w['subject'],
        }


def get_class_weighted_sampler(dataset):
    labels = [w['label'] for w in dataset.windows]
    counts = np.bincount(labels)
    weights = 1.0 / counts
    sample_w = [weights[l] for l in labels]
    return WeightedRandomSampler(sample_w, len(sample_w))

In [8]:
# TRAINING

def train_acbl_isolation(model, acbl_loss_fn, train_loader, val_loader,
                         config, device):
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=config.lr, weight_decay=config.weight_decay)

    train_labels = [w['label'] for w in train_loader.dataset.windows]
    n0 = sum(1 for l in train_labels if l == 0)
    n1 = sum(1 for l in train_labels if l == 1)
    cw = torch.tensor([len(train_labels) / (2*n0),
                        len(train_labels) / (2*n1)],
                       dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=cw)

    best_val_loss = float('inf')
    wait = 0
    history = []

    for epoch in range(config.n_train_epochs):
        if epoch < config.warmup_epochs:
            phase = "warmup"
        elif epoch < config.warmup_epochs + config.formation_epochs:
            phase = "form"
        else:
            phase = "full"
        use_acbl = epoch >= config.warmup_epochs
        detach_bnd = (phase == "form")

        if epoch == config.warmup_epochs + config.formation_epochs:
            wait = 0
            best_val_loss = float('inf')
            print("  >> Full phase: boundaries reconnected, patience reset")

        model.train()
        stats = defaultdict(list)

        for batch in tqdm(train_loader, desc=f'Ep{epoch:2d} [{phase:7s}]',
                          leave=False):
            x = batch['x'].to(device)
            y = batch['label'].to(device)
            ep_labels = batch['epoch_labels'].to(device)

            optimizer.zero_grad()
            out = forward_with_intermediates(model, x,
                                             detach_boundaries=detach_bnd)

            cls_loss = criterion(out['logits'], y) * config.cls_weight
            total_loss = cls_loss

            if use_acbl:
                acbl_out = acbl_loss_fn(
                    out['boundary_probs'], out['encoder_h'],
                    out['regime_attention'], ep_labels, epoch)
                total_loss = total_loss + config.acbl_weight * acbl_out['acbl_total']
                stats['acbl'].append(acbl_out['acbl_total'].item())

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            stats['cls'].append(cls_loss.item())
            with torch.no_grad():
                bnd = out['boundaries']
                stats['bnd_mean'].append(bnd.mean().item())
                stats['bnd_std'].append(bnd.std().item())

        # Validate
        model.eval()
        val_losses = []
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                x = batch['x'].to(device)
                y = batch['label'].to(device)
                out = forward_with_intermediates(model, x, False)
                val_losses.append(criterion(out['logits'], y).item())
                all_preds.extend(out['logits'].argmax(1).cpu().numpy())
                all_labels.extend(y.cpu().numpy())

        val_cls = np.mean(val_losses)
        val_acc = accuracy_score(all_labels, all_preds)

        log = {
            'epoch': epoch, 'phase': phase,
            'train_cls': np.mean(stats['cls']),
            'val_cls': val_cls, 'val_acc': val_acc,
            'bnd_mean': np.mean(stats['bnd_mean']),
            'bnd_std': np.mean(stats['bnd_std']),
        }
        history.append(log)

        print(f"  Ep {epoch:2d} [{phase:7s}] "
              f"cls={log['train_cls']:.4f} val={val_cls:.4f} "
              f"acc={val_acc:.3f} "
              f"bnd_m={log['bnd_mean']:.4f} bnd_s={log['bnd_std']:.4f}")

        if val_cls < best_val_loss:
            best_val_loss = val_cls
            wait = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'epoch': epoch, 'val_loss': val_cls, 'val_acc': val_acc,
            }, config.model_save_dir / 'acbl_isolation_chbmit_best.pt')
        else:
            wait += 1
            if wait >= config.patience:
                print(f"  Early stopping at epoch {epoch}")
                break

    return history

In [9]:
# ONSET EVALUATION

def evaluate_onset_detection(model, test_loader, config, device):
    model.eval()
    results = {'per_window': [], 'classification': {
        'preds': [], 'labels': [], 'probs': []}}

    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Evaluating'):
            x = batch['x'].to(device)
            labels = batch['label']
            subjects = batch['subject']
            ep_labels = batch['epoch_labels']

            out = forward_with_intermediates(model, x, False)
            boundaries = out['boundaries'].cpu().numpy()
            probs = F.softmax(out['logits'], dim=1).cpu().numpy()
            preds = out['logits'].argmax(1).cpu().numpy()

            results['classification']['preds'].extend(preds.tolist())
            results['classification']['labels'].extend(labels.numpy().tolist())
            results['classification']['probs'].extend(probs[:, 1].tolist())

            tpe = config.tokens_per_epoch
            spt = config.seconds_per_token

            for i in range(len(labels)):
                bnd = boundaries[i]
                info = {
                    'subject': subjects[i],
                    'label': int(labels[i]),
                    'pred': int(preds[i]),
                    'prob_seizure': float(probs[i, 1]),
                    'boundary_profile': bnd.tolist(),
                    'boundary_mean': float(bnd.mean()),
                    'boundary_std': float(bnd.std()),
                    'boundary_max': float(bnd.max()),
                    'epoch_labels': ep_labels[i].tolist(),
                }

                # Onset analysis for seizure windows
                if labels[i] == 1:
                    bnd_smooth = gaussian_filter1d(bnd, sigma=3)
                    threshold = bnd.mean() + 1.0 * bnd.std()
                    peaks, props = find_peaks(bnd_smooth,
                                              height=threshold, distance=10)
                    info['n_peaks'] = len(peaks)
                    info['peak_tokens'] = peaks.tolist()
                    info['peak_times_sec'] = (peaks * spt).tolist()
                    info['peak_heights'] = (
                        props['peak_heights'].tolist() if len(peaks) else [])

                    # Find onset token: first token where a seizure epoch starts
                    # Transition: normal->seizure within this window
                    el = ep_labels[i].numpy()
                    onset_token = None
                    for ep in range(len(el)):
                        if el[ep] == 1:
                            if ep == 0 or el[ep - 1] == 0:
                                onset_token = ep * tpe
                                break
                    info['onset_token'] = onset_token

                    if onset_token is not None and len(peaks) > 0:
                        # Latency: distance from onset to nearest peak
                        peak_dists = np.abs(peaks - onset_token)
                        nearest_idx = np.argmin(peak_dists)
                        latency_tokens = peaks[nearest_idx] - onset_token
                        latency_sec = latency_tokens * spt
                        info['latency_tokens'] = int(latency_tokens)
                        info['latency_sec'] = float(latency_sec)

                    # Center vs surround
                    center_s = tpe
                    center_e = 2 * tpe
                    center_bnd = bnd[center_s:center_e]
                    surround_bnd = np.concatenate([bnd[:center_s], bnd[center_e:]])
                    info['center_bnd_mean'] = float(center_bnd.mean())
                    info['surround_bnd_mean'] = float(surround_bnd.mean())
                    info['center_vs_surround'] = float(
                        center_bnd.mean() - surround_bnd.mean())

                results['per_window'].append(info)

    return results


def compute_onset_metrics(eval_results, config):
    seizure_w = [w for w in eval_results['per_window'] if w['label'] == 1]
    normal_w = [w for w in eval_results['per_window'] if w['label'] == 0]

    print(f"\n{'='*60}")
    print("SEIZURE ONSET BOUNDARY EVALUATION")
    print(f"{'='*60}")
    print(f"Windows: {len(seizure_w)} seizure, {len(normal_w)} normal")

    # Boundary activation
    sz_means = [w['boundary_mean'] for w in seizure_w]
    nm_means = [w['boundary_mean'] for w in normal_w]
    sep = np.mean(sz_means) - np.mean(nm_means)
    print(f"\n--- Boundary Activation ---")
    print(f"  Seizure: mean={np.mean(sz_means):.4f}")
    print(f"  Normal:  mean={np.mean(nm_means):.4f}")
    print(f"  Separation: {sep:.4f}")

    # Peaks
    n_peaks = [w.get('n_peaks', 0) for w in seizure_w]
    pct_with_peaks = sum(1 for n in n_peaks if n > 0) / max(len(seizure_w), 1)
    print(f"\n--- Peaks in Seizure Windows ---")
    print(f"  With peaks: {sum(1 for n in n_peaks if n > 0)}/{len(seizure_w)} "
          f"({pct_with_peaks:.1%})")
    print(f"  Avg peaks: {np.mean(n_peaks):.2f}")

    # Latency
    latencies = [w['latency_sec'] for w in seizure_w
                 if 'latency_sec' in w]
    if latencies:
        print(f"\n--- Onset Detection Latency ---")
        print(f"  Mean: {np.mean(latencies):.2f}s")
        print(f"  Median: {np.median(latencies):.2f}s")
        print(f"  Std: {np.std(latencies):.2f}s")
        print(f"  Within 5s: {sum(1 for l in latencies if abs(l) <= 5)}"
              f"/{len(latencies)}")

    # Center vs surround
    deltas = [w['center_vs_surround'] for w in seizure_w
              if 'center_vs_surround' in w]
    if deltas:
        print(f"\n--- Center vs Surround ---")
        print(f"  Mean delta: {np.mean(deltas):.4f}")
        print(f"  Positive: {sum(1 for d in deltas if d > 0)}/{len(deltas)}")

    # Classification
    cls = eval_results['classification']
    y_true = np.array(cls['labels'])
    y_pred = np.array(cls['preds'])
    y_prob = np.array(cls['probs'])

    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro')
    kappa = cohen_kappa_score(y_true, y_pred)
    try:
        auroc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auroc = float('nan')

    cm = confusion_matrix(y_true, y_pred)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    else:
        tn = fp = fn = tp = 0
        sens = spec = 0

    print(f"\n--- Classification ---")
    print(f"  Acc={acc:.4f} F1M={f1m:.4f} Kappa={kappa:.4f}")
    print(f"  AUROC={auroc:.4f} Sens={sens:.4f} Spec={spec:.4f}")
    print(f"  CM: TN={tn} FP={fp} FN={fn} TP={tp}")

    return {
        'boundary_separation': sep,
        'pct_with_peaks': pct_with_peaks,
        'mean_latency_sec': np.mean(latencies) if latencies else None,
        'median_latency_sec': np.median(latencies) if latencies else None,
        'center_vs_surround': np.mean(deltas) if deltas else 0,
        'acc': acc, 'f1m': f1m, 'kappa': kappa, 'auroc': auroc,
        'sensitivity': sens, 'specificity': spec,
    }

In [10]:
# PELT BASELINE COMPARISON

def pelt_onset_evaluation(test_loader, config):
    """
    Run PELT on raw EEG for each seizure window and measure
    onset detection latency for comparison with learned boundaries.
    """
    try:
        import ruptures as rpt
    except ImportError:
        print("pip install ruptures")
        return None

    results = []

    for batch in tqdm(test_loader, desc='PELT baseline'):
        x = batch['x'].numpy()        # (B, n_ctx, C, T)
        labels = batch['label'].numpy()
        ep_labels = batch['epoch_labels'].numpy()

        for i in range(len(labels)):
            if labels[i] != 1:
                continue

            # Concatenate 3 epochs: (3, C, T) -> (C, 3*T)
            signal = x[i].reshape(x.shape[2], -1)  # (C, 9000)
            mean_sig = signal.mean(axis=0)  # (9000,)

            try:
                algo = rpt.Pelt(model="l2", jump=10, min_size=50).fit(mean_sig)
                cps = algo.predict(pen=10)
                cp_times = [cp / config.sfreq for cp in cps[:-1]]
            except Exception:
                cp_times = []

            # Find onset token from epoch labels
            el = ep_labels[i]
            onset_sec = None
            for ep in range(len(el)):
                if el[ep] == 1 and (ep == 0 or el[ep-1] == 0):
                    onset_sec = ep * config.epoch_duration
                    break

            latency = None
            if onset_sec is not None and cp_times:
                dists = [abs(t - onset_sec) for t in cp_times]
                nearest_idx = np.argmin(dists)
                latency = cp_times[nearest_idx] - onset_sec

            results.append({
                'n_changepoints': len(cp_times),
                'cp_times_sec': cp_times,
                'onset_sec': onset_sec,
                'latency_sec': latency,
            })

    if results:
        latencies = [r['latency_sec'] for r in results
                     if r['latency_sec'] is not None]
        avg_cps = np.mean([r['n_changepoints'] for r in results])
        print(f"\n--- PELT Baseline ---")
        print(f"  Avg changepoints/window: {avg_cps:.1f}")
        if latencies:
            print(f"  Onset latency mean: {np.mean(latencies):.2f}s")
            print(f"  Onset latency median: {np.median(latencies):.2f}s")
            print(f"  Within 5s: {sum(1 for l in latencies if abs(l) <= 5)}"
                  f"/{len(latencies)}")

    return results

In [11]:
# VISUALIZATION

def plot_boundary_profiles(eval_results, config, save_dir):
    seizure_w = [w for w in eval_results['per_window'] if w['label'] == 1]
    normal_w = [w for w in eval_results['per_window'] if w['label'] == 0]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    spt = config.seconds_per_token

    # Avg seizure profile
    ax = axes[0, 0]
    if seizure_w:
        profiles = np.array([w['boundary_profile'] for w in seizure_w[:100]])
        mean_p = profiles.mean(axis=0)
        std_p = profiles.std(axis=0)
        t = np.arange(len(mean_p)) * spt
        ax.plot(t, mean_p, 'r-', lw=1.5, label='Mean')
        ax.fill_between(t, mean_p - std_p, mean_p + std_p,
                        alpha=0.2, color='red')
        ax.axvspan(30, 60, alpha=0.1, color='orange', label='Center epoch')
        ax.set_title('Seizure Windows: Avg Boundary Profile')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Boundary activation')
        ax.legend()

    # Avg normal profile
    ax = axes[0, 1]
    if normal_w:
        profiles = np.array([w['boundary_profile'] for w in normal_w[:100]])
        mean_p = profiles.mean(axis=0)
        std_p = profiles.std(axis=0)
        t = np.arange(len(mean_p)) * spt
        ax.plot(t, mean_p, 'b-', lw=1.5, label='Mean')
        ax.fill_between(t, mean_p - std_p, mean_p + std_p,
                        alpha=0.2, color='blue')
        ax.set_title('Normal Windows: Avg Boundary Profile')
        ax.set_xlabel('Time (s)')
        ax.legend()

    # Distribution comparison
    ax = axes[1, 0]
    ax.hist([w['boundary_mean'] for w in normal_w], bins=30,
            alpha=0.6, label='Normal', color='blue')
    ax.hist([w['boundary_mean'] for w in seizure_w], bins=30,
            alpha=0.6, label='Seizure', color='red')
    ax.set_title('Boundary Mean Distribution')
    ax.legend()

    # Individual seizure examples
    ax = axes[1, 1]
    for j, w in enumerate(seizure_w[:5]):
        bnd = np.array(w['boundary_profile'])
        t = np.arange(len(bnd)) * spt
        ax.plot(t, bnd, alpha=0.7, label=f"Win {j+1}")
    ax.axvspan(30, 60, alpha=0.1, color='orange')
    ax.set_title('Individual Seizure Windows')
    ax.set_xlabel('Time (s)')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(save_dir / 'boundary_onset_profiles.png', dpi=150)
    plt.close()
    print(f"Saved: {save_dir / 'boundary_onset_profiles.png'}")

In [12]:
# MAIN

def main():
    print("=" * 60)
    print("CHB-MIT SEIZURE ONSET BOUNDARY EVALUATION")
    print("=" * 60)

    # Load
    epochs, labels, subject_ids, subj_sz = load_chbmit_data(Config.h5_path)

    # Split
    print("\nSubject split:")
    splits = round_robin_split(subj_sz, Config.seed)

    # Datasets
    print("\nBuilding datasets:")
    train_ds = CHBMITOnsetDataset(epochs, labels, subject_ids,
                                   splits['train'], Config.n_context)
    val_ds = CHBMITOnsetDataset(epochs, labels, subject_ids,
                                 splits['val'], Config.n_context)
    test_ds = CHBMITOnsetDataset(epochs, labels, subject_ids,
                                  splits['test'], Config.n_context)

    train_sampler = get_class_weighted_sampler(train_ds)
    train_loader = DataLoader(train_ds, batch_size=Config.batch_size,
                              sampler=train_sampler, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=Config.batch_size,
                            shuffle=False, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=Config.batch_size,
                             shuffle=False, num_workers=0)

    # Model
    model = MultiResContrastiveNeuroState(
        n_channels=Config.n_channels,
        n_samples=Config.n_samples,
        n_classes=Config.n_classes,
        embed_dim=Config.embed_dim,
        n_layers=Config.n_layers,
        dropout=Config.dropout,
        contrast_scales=Config.contrast_scales,
        cp_hidden=Config.cp_hidden,
        n_context_epochs=Config.n_context,
    ).to(device)

    # Verify tokens_per_epoch
    print(f"\nVerified: tokens_per_epoch = {model.tokens_per_epoch}")
    assert model.tokens_per_epoch == Config.tokens_per_epoch, \
        f"Mismatch: model={model.tokens_per_epoch}, config={Config.tokens_per_epoch}"

    # ACBL loss
    acbl_loss_fn = ACBLLoss(
        tokens_per_epoch=model.tokens_per_epoch,
        n_epochs=Config.n_context,
    )

    # Train
    print("\nTraining ACBL + Isolation v1:")
    history = train_acbl_isolation(model, acbl_loss_fn, train_loader,
                                   val_loader, Config, device)

    # Load best
    ckpt = torch.load(Config.model_save_dir / 'acbl_isolation_chbmit_best.pt',
                  map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded best from epoch {ckpt['epoch']} (val_acc={ckpt['val_acc']:.4f})")

    # Evaluate
    print("\nOnset boundary evaluation:")
    eval_results = evaluate_onset_detection(model, test_loader, Config, device)
    metrics = compute_onset_metrics(eval_results, Config)

    # Visualize
    plot_boundary_profiles(eval_results, Config, Config.figures_dir)

    # PELT baseline
    print("\nPELT baseline:")
    pelt_results = pelt_onset_evaluation(test_loader, Config)

    # Save
    save_path = Config.model_save_dir / 'onset_eval_results.json'
    with open(save_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"\nResults saved to {save_path}")

    # Save history
    hist_path = Config.model_save_dir / 'acbl_chbmit_history.json'
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f"History saved to {hist_path}")


if __name__ == "__main__":
    main()

CHB-MIT SEIZURE ONSET BOUNDARY EVALUATION
Loaded: 9639 epochs, 17 ch, 100 Hz
  Seizure: 312, Normal: 9327
    chb15: 73 seizure epochs
    chb12: 54 seizure epochs
    chb08: 35 seizure epochs
    chb05: 34 seizure epochs
    chb01: 24 seizure epochs
    chb03: 22 seizure epochs
    chb10: 19 seizure epochs
    chb20: 13 seizure epochs
    chb14: 12 seizure epochs
    chb17: 10 seizure epochs
    chb22: 9 seizure epochs
    chb19: 7 seizure epochs

Subject split:
  train: ['chb15', 'chb12', 'chb01', 'chb03', 'chb14', 'chb17'] (195 seizure epochs)
  val: ['chb08', 'chb10', 'chb22'] (63 seizure epochs)
  test: ['chb05', 'chb20', 'chb19'] (54 seizure epochs)

Building datasets:
  4671 windows (193 seizure, 4478 normal)
  3448 windows (63 seizure, 3385 normal)
  1496 windows (53 seizure, 1443 normal)
MultiResContrastiveNeuroState: 1.07M params, 588 tokens (196/epoch)

Verified: tokens_per_epoch = 196

Training ACBL + Isolation v1:
  Ep  0 [warmup ] cls=0.3503 val=2.5201 acc=0.018 bnd_m=0.4

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=30d601c2-0a51-44b0-ac6c-72cf48d1679e' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>